In [1]:
pip install sqlalchemy pandas psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
from sqlalchemy import create_engine

DB_USER = "postgres"
DB_PASS = "akila123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "financial_risk"

connection_string = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

query = "SELECT * FROM transactions;"
df = pd.read_sql(query, engine)

print("DataFrame Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())
print("\nDataFrame Info:")
print(df.info())

DataFrame Shape: (555719, 23)

First 5 Rows:
   csv_index trans_date_trans_time            cc_num  \
0          0   2020-06-21 12:14:25  2291163933867244   
1          1   2020-06-21 12:14:33  3573030041201292   
2          2   2020-06-21 12:14:53  3598215285024754   
3          3   2020-06-21 12:15:15  3591919803438423   
4          4   2020-06-21 12:15:17  3526826139003047   

                               merchant        category    amt   first  \
0                 fraud_Kirlin and Sons   personal_care   2.86    Jeff   
1                  fraud_Sporer-Keebler   personal_care  29.84  Joanne   
2  fraud_Swaniawski, Nitzsche and Welch  health_fitness  41.28  Ashley   
3                     fraud_Haley Group        misc_pos  60.05   Brian   
4                 fraud_Johnston-Casper          travel   3.19  Nathan   

       last gender                       street  ...      lat      long  \
0   Elliott      M            351 Darlene Green  ...  33.9659  -80.9355   
1  Williams      F     

In [6]:
import numpy as np

def haversine_np(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees) in miles.
    """

    # 1. Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # 2. Compute differences
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # 3. Apply Haversine formula
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))

    # 4. Multiply by Earth radius in miles (3,958.8 miles)
    miles = 3958.8 * c
    return miles

In [7]:
# 1. Geodesic Distance
df['distance_miles'] = haversine_np(df['lat'], df['long'], df['merch_lat'], df['merch_long'])

# 2. Customer Age
df['dob'] = pd.to_datetime(df['dob'])
df['age'] = (pd.to_datetime('today') - df['dob']).dt.days // 365

# 3. Time Features
df['hour_of_day'] = df['trans_date_trans_time'].dt.hour
df['day_of_week'] = df['trans_date_trans_time'].dt.day_name()

# 4. Verify engineered features
df[['lat', 'long', 'merch_lat', 'merch_long', 'distance_miles', 'age', 'hour_of_day', 'day_of_week']].head()

,lat,long,merch_lat,merch_long,distance_miles,age,hour_of_day,day_of_week
0,33.9659,-80.9355,33.986391,-81.200714,15.261955,58,12,Sunday
1,40.3207,-110.4360,39.450498,-109.960431,65.198157,36,12,Sunday
2,40.6729,-73.5365,40.495810,-74.196111,36.711068,55,12,Sunday
3,28.5697,-80.8191,28.812398,-80.883061,17.211284,39,12,Sunday
4,44.2529,-85.0170,44.959148,-85.884734,64.831552,71,12,Sunday


In [8]:
# Compare distance metrics between legitimate and fraudulent transactions
df.groupby('is_fraud')['distance_miles'].agg(
    count='count',
    mean_miles='mean',
    median_miles='median',
    max_miles='max'
).round(2)

,count,mean_miles,median_miles,max_miles
is_fraud,,,,
0,553574,47.29,48.58,93.78
1,2145,47.36,48.84,88.76


In [9]:
# Export the cleaned, feature-engineered dataset for Power BI
df.to_csv('cleaned_financial_risk_data.csv', index=False)
print("Export complete!")

Export complete!
